# SleepEDF Knowledge Distillation Project

This notebook provides an end-to-end pipeline for training and evaluating a lightweight **Sleep Stage Classification** student model on the **SleepEDF dataset (2-channel EEG/EOG)** using Knowledge Distillation to achieve high efficiency without sacrificing performance.

---

### Project Highlights & Key Features

* **Class Imbalance Handling**: Utilizes `WeightedRandomSampler` and class-weighted Cross-Entropy loss to address severe sleep stage class imbalances.
* **Knowledge Distillation (KD)**:
  * Leverages soft logits from a high-capacity pretrained **Teacher Model** (`CNNTransformerTeacher`).
  * Transfers knowledge to a compact **Student Model** (`SleepStudentCNN`) for efficient model compression.
* **Robust Evaluation & Logging**:
  * Benchmarks performance across Teacher, Vanilla Student, and KD Student models (Val Loss, Val Acc, Val F1, Model Size).
  * Automatically exports experimental metrics to `summary.csv` and `history.csv`, followed by generating a consolidated visualization dashboard.

---

### Pipeline Workflow

1. **Hardware & Environment Check**: Verifies PyTorch GPU setup (CUDA 12.1) and runtime environment.
2. **Configuration & Path Setup**: Loads JSON configs and creates dynamic output directories.
3. **Data Loading & Preprocessing**: Loads `full_dataset.pt`, handles Train/Val/Test splitting, and configures class-weighted sampling.
4. **Teacher Evaluation**: Loads and evaluates the pretrained Teacher model (`best_teacher.pth`).
5. **Vanilla Student Evaluation**: Evaluates the baseline Student model trained without KD (`best_student_vanilla.pth`).
6. **Student KD Training**: Trains the Student model guided by Teacher soft targets and KL divergence loss.
7. **Consolidation & CSV Export**: Aggregates histories, metrics, and predictions across all three models into CSV files.
8. **Dashboard Visualization**: Generates and displays comparative plots for training curves, F1 scores, and model footprint.

In [ ]:
# [0. Necessary library imports]

# Package installation (Run once if necessary)
# !pip install --target="{LIB_PATH}" --upgrade mne
# !pip install --target="{LIB_PATH}" --upgrade torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
import os, sys
import torch

# ==============================================================================
# 1. GPU Hardware Check
# ==============================================================================
if not torch.cuda.is_available():
    print("⚠️ CUDA is not available. Attempting to reinstall PyTorch GPU version...")
    # Force reinstall PyTorch + CUDA 12.1 tailored for Colab environment
    !pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    
    print("\n🔄 Package reinstallation complete. Please click [Runtime] -> [Restart session] in Colab menu and run this cell again!")
    sys.exit()

# ==============================================================================
# 2. PyTorch CUDA & Device Setup
# ==============================================================================
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Available GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Capacity: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Currently Set Device: {device}")

In [ ]:
# [1. Module Import & Configuration Loading]
import datetime
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader, random_split, WeightedRandomSampler
import torch.nn as nn

from src import UnifiedTrainer, visualize_from_csv, save_metrics_to_csv, create_warmup_cosine_scheduler
from src.models.teacher import CNNTransformerTeacher
from src.models.student import SleepStudentCNN
from src.losses.kd_loss import KDLoss, compute_class_weights
from src.utils import get_model_size_mb

from config import Config

Config.load_from_json("config.json")

In [ ]:
# [2. Directory & Path Setup]
current_date = datetime.datetime.now().strftime("%Y%m%d")
trial_num = Config.TRIAL_NUM

LIB_PATH = os.getenv('YOUR_LIB_PATH')
PROJ_DIR = os.getenv('PROJ_DIR')
print(f"Loaded Path: {LIB_PATH}")
print(f"Project Directory: {PROJ_DIR}")

if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

OUTPUT_DIR = os.path.join(PROJ_DIR, "sleepEDF_outputs")
SAVE_DIR = os.path.join(OUTPUT_DIR, current_date, f"trial_{trial_num:02d}")
SAVE_DIR_VANILLA = SAVE_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

Config.load_from_json(os.path.join(PROJ_DIR, "config.json"))

In [ ]:
# =====================================================================
# 3. Data Loading, Sampling, Data Split & DataLoader Construction
# =====================================================================

print("[1/4] Loading Sleep-EDF dataset from .pt file")
pt_file_path = os.path.join(OUTPUT_DIR, "full_dataset.pt")
loaded_data = torch.load(pt_file_path, mmap=True, weights_only=True)

print("[2/4] Copying random dataset subset")
subset_size = 50000

signals_subset = loaded_data["signals"][:subset_size].clone().detach() # Sleep-EDF signal data
labels_subset = loaded_data["labels"][:subset_size].clone().detach()   # Sleep stage label
full_dataset = TensorDataset(signals_subset, labels_subset)

print(f"Sleep-EDF dataset prepared, size: {len(full_dataset)}")

# Data splitting (Train/Val/Test)
print("[3/4] Splitting train/val/test datasets")
total_len = len(full_dataset)
train_len = int(0.8 * total_len)
val_len = int(0.1 * total_len)
test_len = total_len - train_len - val_len

train_ds, val_ds, test_ds = random_split(full_dataset, [train_len, val_len, test_len])
print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

# Class-weighted sampling for train dataset
print("[4/4] Applying class-weighted sampling based on sleep stage")
train_indices = train_ds.indices
train_labels_tensor = labels_subset[train_indices].long()
class_weights = compute_class_weights(train_labels_tensor, num_classes=Config.NUM_CLASSES, device=device, method='sqrt').to(device)
sample_weights = class_weights[train_labels_tensor].float()

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoader definitions
train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("DataLoader initialization completed.")

In [ ]:
# =====================================================================
# 4. Load Pretrained Teacher Model & Evaluate
# =====================================================================

# Initialize Teacher model
teacher_model = CNNTransformerTeacher(
    in_channels=7,
    num_classes=Config.NUM_CLASSES,
    teacher_embedding_dim=Config.TEACHER_EMBEDDING_DIM,
    teacher_nhead=Config.TEACHER_NHEAD,
    teacher_dropout=Config.TEACHER_DROPOUT,
    teacher_num_layers=Config.TEACHER_NUM_LAYERS
).to(device)

# Load best Teacher checkpoint (.pth)
teacher_checkpoint_path = os.path.join(SAVE_DIR, "best_teacher.pth")
if os.path.exists(teacher_checkpoint_path):
    teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location=device, weights_only=False)
    teacher_model.load_state_dict(teacher_checkpoint['model_state_dict'])
    print(f"✅ Teacher model weights loaded successfully! (Epoch {teacher_checkpoint.get('epoch', '?')}, Val F1: {teacher_checkpoint.get('val_f1', 0.0):.4f})")
else:
    raise FileNotFoundError(f"❌ Teacher checkpoint file not found: {teacher_checkpoint_path}")

# Freeze Teacher parameters and set to evaluation mode
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

# Evaluate Teacher model
trainer_teacher = UnifiedTrainer(
    model=teacher_model,
    eval_criterion=KDLoss(alpha=0.0),
    device=device
)

_, _, teacher_f1, true_labels, teacher_preds = trainer_teacher.evaluate(val_loader, is_student=False)

teacher_history = teacher_checkpoint.get('history', {})

t_train_loss = teacher_history.get('train_loss', [])
t_val_loss   = teacher_history.get('val_loss', [])
t_train_acc  = teacher_history.get('train_acc', [])
t_val_acc    = teacher_history.get('val_acc', [])
t_val_f1     = teacher_history.get('val_f1', [])
t_lr         = teacher_history.get('lr', [])

t_f1_best = teacher_checkpoint.get('val_f1', 0.0)
t_epoch_best = teacher_checkpoint.get('epoch', 0)

print(f"✅ Teacher Best Model (Epoch {t_epoch_best}) - Val F1: {t_f1_best:.4f}")
print(f"📊 Val Acc History ({len(t_val_acc)} total epochs): {t_val_acc[:5]} ... {t_val_acc[-1:]}")
print(f"📊 Val Loss History ({len(t_val_loss)} total epochs): {t_val_loss[:5]} ... {t_val_loss[-1:]}")

In [ ]:
# =====================================================================
# 5. Load Pretrained Vanilla Student Model & Evaluate
# =====================================================================

student_vanilla_model = SleepStudentCNN(
    in_channels=2,
    num_classes=Config.NUM_CLASSES
).to(device)

vanilla_checkpoint_path = os.path.join(SAVE_DIR, "best_student_vanilla.pth")

if os.path.exists(vanilla_checkpoint_path):
    vanilla_checkpoint = torch.load(vanilla_checkpoint_path, map_location=device, weights_only=False)
    student_vanilla_model.load_state_dict(vanilla_checkpoint['model_state_dict'])
    print(f"✅ Vanilla Student model weights loaded successfully! (Epoch {vanilla_checkpoint.get('epoch', '?')}, Val F1: {vanilla_checkpoint.get('val_f1', 0.0):.4f})")
else:
    raise FileNotFoundError(f"❌ Vanilla Student checkpoint file not found: {vanilla_checkpoint_path}")

vanilla_history = vanilla_checkpoint.get('history', {})

v_train_loss = vanilla_history.get('train_loss', [])
v_val_loss   = vanilla_history.get('val_loss', [])
v_train_acc  = vanilla_history.get('train_acc', [])
v_val_acc    = vanilla_history.get('val_acc', [])
v_val_f1     = vanilla_history.get('val_f1', [])
v_lr         = vanilla_history.get('lr', [])

v_f1_best = vanilla_checkpoint.get('val_f1', 0.0)
v_epoch_best = vanilla_checkpoint.get('epoch', 0)

# Evaluate Vanilla Student model
trainer_vanilla = UnifiedTrainer(
    model=student_vanilla_model,
    eval_criterion=KDLoss(alpha=0.0),
    device=device
)

_, _, vanilla_f1, _, vanilla_preds = trainer_vanilla.evaluate(val_loader, is_student=True)

print(f"✅ Vanilla Student Best Model (Epoch {v_epoch_best}) - Val F1: {v_f1_best:.4f}")
print(f"📊 Val Acc History ({len(v_val_acc)} total epochs): {v_val_acc[:5]} ... {v_val_acc[-1:]}")
print(f"📊 Val Loss History ({len(v_val_loss)} total epochs): {v_val_loss[:5]} ... {v_val_loss[-1:]}")

In [ ]:
# =====================================================================
# 6. Knowledge Distillation (KD) Student Model Training
# =====================================================================

student_kd_model = SleepStudentCNN(
    in_channels=Config.STUDENT_IN_CHANNELS,
    num_classes=Config.NUM_CLASSES
).to(device)

optimizer_kd = torch.optim.AdamW(
    student_kd_model.parameters(),
    lr=Config.STUDENT_LR,
    weight_decay=Config.WEIGHT_DECAY_STUDENT,
    eps=1e-6
)

scheduler_kd = create_warmup_cosine_scheduler(
    optimizer_kd, 
    epochs=Config.STUDENT_EPOCHS, 
    warmup_epochs=3,
    start_factor=0.01,
    min_lr=1e-6
)

custom_class_weights = torch.tensor([1.0, 4.80, 2.00, 3.50, 4.00], device=device)

criterion_kd_train = KDLoss(
    alpha=Config.KD_ALPHA,
    temperature=Config.KD_TEMPERATURE,
    ce_weight=class_weights,
    kl_weight=custom_class_weights
)

criterion_kd_val = KDLoss(
    alpha=0.0,
    ce_weight=None,
    kl_weight=None
)

trainer_kd = UnifiedTrainer(
    model=student_kd_model,
    teacher_model=teacher_model,
    optimizer=optimizer_kd,
    train_criterion=criterion_kd_train,
    eval_criterion=criterion_kd_val,
    device=device,
    scheduler=scheduler_kd
)

trainer_kd.fit(
    train_loader,
    val_loader,
    epochs=Config.STUDENT_EPOCHS,
    model_name="student_kd",
    is_student=True
)

_, _, kd_f1, _, kd_preds = trainer_kd.evaluate(val_loader, is_student=True)
kd_history = trainer_kd.history

In [ ]:
# =====================================================================
# 7. Consolidate Histories & Predictions
# =====================================================================

# Group training histories into a single dictionary
histories = {
    'teacher': teacher_checkpoint.get('history', {}),
    'vanilla': vanilla_checkpoint.get('history', {}),
    'kd': trainer_kd.history
}

# Group predictions into a single dictionary
preds = {
    'teacher': teacher_preds,
    'vanilla': vanilla_preds,
    'kd': kd_preds
}

In [ ]:
# =====================================================================
# 8. Save Metrics & Histories to CSV
# =====================================================================

summary_csv_path = os.path.join(SAVE_DIR, f"sleepedf_trial_{Config.TRIAL_NUM:02d}_summary.csv")
history_csv_path = os.path.join(SAVE_DIR, f"sleepedf_trial_{Config.TRIAL_NUM:02d}_history.csv")

teacher_size_mb = get_model_size_mb(teacher_model)
student_size_mb = get_model_size_mb(student_vanilla_model)

# Export metrics and history logs to CSV files
save_metrics_to_csv(
    summary_csv_path=summary_csv_path,
    history_csv_path=history_csv_path,
    cfg_obj=Config,
    teacher_size_mb=teacher_size_mb,
    student_size_mb=student_size_mb,
    histories=histories,
    preds=preds,
    true_labels=true_labels
)

In [ ]:
# [8. Save Dashboard Visualization]
SAVE_DIR = os.path.join(OUTPUT_DIR, "20260904", f"trial_{Config.TRIAL_NUM:02d}")
summary_csv_path = os.path.join(SAVE_DIR, f"sleepedf_trial_{Config.TRIAL_NUM:02d}_summary.csv")
history_csv_path = os.path.join(SAVE_DIR, f"sleepedf_trial_{Config.TRIAL_NUM:02d}_history.csv")

visualize_from_csv(
    summary_csv_path=summary_csv_path,
    history_csv_path=history_csv_path,
    save_fig=True
)